# Bader Charge Analysis of CuTA and MgTA MOFs

Compares Bader charges of the hydrated and dehydrated metal-triazolate MOFs M(TA)2 (M = Cu, Mg) to quantify charge redistribution on water adsorption.

**Workflow**

1. Read atom order and element counts from `POSCAR_*`.
2. Read per-atom Bader charges from `ACF_*.dat` (Henkelman group `bader` code).
3. Convert to net atomic charge: `net = reference electrons - Bader charge`.
4. Split the hydrated structure into framework atoms (present in both states) and water atoms (hydrated only).
5. Compare framework atoms between states, and report the charge held by water.
6. Write CSVs to `results/`.

**Expected layout** (run from `notebooks/` or the repo root)

```
data/cuta/  POSCAR_hydr  POSCAR_dehy  ACF_hydr_CuTA.dat  ACF_dehy_CuTA.dat
data/mgta/  POSCAR_hydr  POSCAR_dehy  ACF_hydr_MgTA.dat  ACF_dehy_MgTA.dat
```

**Assumption:** water atoms are appended after the framework atoms, and the framework atom order is identical in the hydrated and dehydrated POSCARs.

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Repo paths (works from notebooks/ or from the repo root)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

# Systems to analyze: folder name -> file tag used in the ACF file names
SYSTEMS = {"cuta": "CuTA", "mgta": "MgTA"}

## 2. Reference electron counts

Net charge is `reference - Bader charge`, so the reference must match how the charge density was built.

| Bader input | Reference per atom |
|---|---|
| `CHGCAR_sum` from `AECCAR0 + AECCAR2` (core + valence, `-ref` flag) | atomic number Z |
| `CHGCAR` only (valence electrons) | `ZVAL` from `OUTCAR` |

The default below is atomic number Z. Section 4 checks that the reference is consistent with the ACF output, so a wrong choice is caught before any results are written.

In [ ]:
# Reference electrons per atom. Default: atomic number Z (all-electron Bader).
# If you ran Bader on valence-only CHGCAR, replace with ZVAL from OUTCAR:
#   grep ZVAL OUTCAR
REFERENCE = {"Cu": 29, "Mg": 12, "C": 6, "N": 7, "O": 8, "H": 1}

## 3. Parsers

`parse_poscar` expands the species and count lines into one element label per atom, in file order. This is the same order the Bader code uses, so atom `i` in the ACF file is atom `i` in the POSCAR.

`parse_acf` reads the 7-column table (`#, X, Y, Z, CHARGE, MIN DIST, ATOMIC VOL`) and ignores the header and footer lines.

In [ ]:
def parse_poscar(path):
    """Return a list with one element symbol per atom, in POSCAR order (VASP5 format)."""
    lines = Path(path).read_text().splitlines()
    species = lines[5].split()
    if not all(s.isalpha() for s in species):
        raise ValueError(f"{path}: no element line found (VASP4 format?). Add symbols on line 6.")
    counts = [int(x) for x in lines[6].split()]
    return [el for el, n in zip(species, counts) for _ in range(n)]


def parse_acf(path):
    """Return a DataFrame of per-atom Bader charges from an ACF.dat file."""
    rows = []
    for line in Path(path).read_text().splitlines():
        p = line.split()
        if len(p) != 7:
            continue
        try:
            rows.append({"atom_id": int(p[0]), "bader_charge": float(p[4])})
        except ValueError:
            continue  # header or separator line
    return pd.DataFrame(rows)

## 4. Build the charge table for one structure

Each atom gets an element label, a net charge, and an index within its element (`el_idx`). `el_idx` is used later to pair atoms between the two states.

The neutrality check sums the net charge over the whole cell. For a neutral cell it should be near zero (a small residual is normal). A large value means the reference electron counts do not match the Bader input.

In [ ]:
def build_table(acf_path, poscar_path, label, tol=0.5):
    elements = parse_poscar(poscar_path)
    df = parse_acf(acf_path)

    if len(df) != len(elements):
        raise ValueError(f"{label}: ACF has {len(df)} atoms, POSCAR has {len(elements)}.")

    df["element"] = elements
    missing = set(elements) - set(REFERENCE)
    if missing:
        raise KeyError(f"{label}: add {sorted(missing)} to REFERENCE.")

    df["net_charge"] = df["element"].map(REFERENCE) - df["bader_charge"]
    df["el_idx"] = df.groupby("element").cumcount()

    total = df["net_charge"].sum()
    status = "OK" if abs(total) < tol else "CHECK REFERENCE ELECTRONS"
    print(f"{label:<11} atoms = {len(df):>4}   total net charge = {total:+.4f} e   [{status}]")
    return df

## 5. Compare hydrated and dehydrated

Atoms are paired by `(element, el_idx)`.

- **Framework atoms** exist in both structures. Their charges are compared directly.
- **Water atoms** exist only in the hydrated structure.

Comparing matched atoms only keeps added water H from inflating the framework H average. The sum over water atoms is the charge held by the adsorbed water. By charge conservation, the framework change should be equal and opposite.

In [ ]:
def compare(dehy, hydr):
    key = ["element", "el_idx"]
    pairs = dehy[key + ["atom_id", "bader_charge", "net_charge"]].merge(
        hydr[key + ["atom_id", "bader_charge", "net_charge"]],
        on=key, suffixes=("_dehy", "_hydr"), how="inner",
    )
    pairs["delta_net"] = pairs["net_charge_hydr"] - pairs["net_charge_dehy"]

    # Hydrated atoms with no partner in the dehydrated structure
    paired = hydr.set_index(key).index.isin(pairs.set_index(key).index)
    water = hydr.loc[~paired, ["atom_id", "element", "bader_charge", "net_charge"]]
    return pairs, water


def element_summary(pairs):
    g = pairs.groupby("element")
    out = pd.DataFrame({
        "n_atoms": g.size(),
        "dehydrated (e)": g["net_charge_dehy"].mean(),
        "hydrated (e)": g["net_charge_hydr"].mean(),
        "delta (hydr - dehy)": g["delta_net"].mean(),
    })
    return out.round(4)

## 6. Run both systems

Prints the per-element table for framework atoms, the water atom charges, and a charge-conservation check. Positive delta means the atom becomes more cationic after water adsorption.

In [ ]:
def analyze(folder, tag):
    d = DATA / folder
    print(f"\n{'=' * 60}\n{tag}\n{'=' * 60}")
    dehy = build_table(d / f"ACF_dehy_{tag}.dat", d / "POSCAR_dehy", "Dehydrated")
    hydr = build_table(d / f"ACF_hydr_{tag}.dat", d / "POSCAR_hydr", "Hydrated")

    pairs, water = compare(dehy, hydr)
    summary = element_summary(pairs)

    print("\nFramework atoms (matched between states)")
    print(summary.to_string())

    print(f"\nWater atoms ({len(water)} atoms, hydrated only)")
    print(water.groupby("element")["net_charge"].agg(["count", "mean"]).round(4).to_string())

    q_water = water["net_charge"].sum()
    q_frame = pairs["delta_net"].sum()
    print(f"\nCharge on water      : {q_water:+.4f} e")
    print(f"Framework change     : {q_frame:+.4f} e")
    print(f"Sum (should be ~0)   : {q_water + q_frame:+.4f} e")

    summary.to_csv(RESULTS / f"{folder}_charges_comparison.csv")
    pairs.round(4).to_csv(RESULTS / f"{folder}_per_atom_charges.csv", index=False)
    water.round(4).to_csv(RESULTS / f"{folder}_water_charges.csv", index=False)
    return pairs, water, summary


results = {folder: analyze(folder, tag) for folder, tag in SYSTEMS.items()}

## 7. Per-atom detail

Atoms with the largest charge change between states, per system. Useful for finding which framework atoms interact most with water.

In [ ]:
for folder, (pairs, water, summary) in results.items():
    top = pairs.reindex(pairs["delta_net"].abs().sort_values(ascending=False).index).head(10)
    print(f"\n{SYSTEMS[folder]}: 10 largest |delta| atoms")
    print(top[["element", "atom_id_dehy", "atom_id_hydr", "net_charge_dehy",
               "net_charge_hydr", "delta_net"]].round(4).to_string(index=False))